In [1]:
import pandas as pd
df=pd.read_csv("BL--ExpressionData.csv")
# 转置并重新设置列名
cols = df["Unnamed: 0"]
df = df.T
df = df.drop(["Unnamed: 0"])
df.columns = cols
df.iloc[:,0]
df.to_csv("ExpressionData.csv",index=False)

In [2]:
#生成pathway zero
import pandas as pd
a=pd.read_csv("ExpressionData.csv")
b=pd.read_csv("pathway_all.csv")
cols=list(a.columns)
cols1 = [i.capitalize() for i in cols]
idx=list(b["Unnamed: 0"])
d = pd.DataFrame(columns=cols1,index=idx)
d.index=idx
d=d.fillna(0)
d
d.to_csv("pathway_zero.csv",index=True)


/tmp/ipykernel_3450/4121569121.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  d=d.fillna(0)


In [3]:
#生成pathway
import pandas as pd
p = pd.read_csv("/home/featurize/work/cs-main/DeepTGI/dataset/pathway/mmu00001 kegg mmuid genename.txt",sep="\t")
a=pd.read_csv("pathway_zero.csv",index_col=0)
print(a)
genes = list(a.columns)
for i in genes:
    keggs=p.query("name==@i")
    if((keggs.empty)):
        a.loc["No",i]=1
    else:
        for j in (list(keggs['mmuid'])):
            a.loc[j,i]=1
print(a)
a.to_csv("pathway_hESC.csv",index=False)

          Btg2  Mia3  Gins2  Gypb  Man1a1  Foxp1  Prmt6  Etv3  Rgpd2  Plod2  \
mmu00010     0     0      0     0       0      0      0     0      0      0   
mmu00020     0     0      0     0       0      0      0     0      0      0   
mmu00030     0     0      0     0       0      0      0     0      0      0   
mmu00040     0     0      0     0       0      0      0     0      0      0   
mmu00051     0     0      0     0       0      0      0     0      0      0   
...        ...   ...    ...   ...     ...    ...    ...   ...    ...    ...   
mmu99996     0     0      0     0       0      0      0     0      0      0   
mmu99997     0     0      0     0       0      0      0     0      0      0   
mmu99998     0     0      0     0       0      0      0     0      0      0   
mmu99999     0     0      0     0       0      0      0     0      0      0   
No           0     0      0     0       0      0      0     0      0      0   

          ...  Pcdh7  Trip13  Pbxip1  Rorb  Gdap1  

In [5]:
#转为大写
import pandas as pd
a=pd.read_csv("pathway_hESC.csv")
col = list(a.columns)
cols = [i.upper() for i in col]
a.columns=cols
a.to_csv("pathway_hESC.csv",index=False)

In [2]:
import pandas as pd
import numpy as np

import pandas as pd
a=pd.read_csv("ExpressionData.csv")
column_names = list(a.columns)
print(len(column_names))

def gmt_to_binary_matrix_with_orphan_flag(gmt_path, target_genes):
    # 1. 预处理：统一转大写
    target_genes_upper = [str(g).upper() for g in target_genes]
    
    pathway_dict = {}
    with open(gmt_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 3:
                continue
            pathway_name = parts[0]
            # 统一转大写匹配
            genes_in_pathway = set([g.upper() for g in parts[2:]])
            pathway_dict[pathway_name] = genes_in_pathway

    print(f"解析完成，共包含 {len(pathway_dict)} 条通路。")

    # 2. 构建二值矩阵
    matrix_data = []
    pathway_names = []

    for name, genes in pathway_dict.items():
        row = [1 if g in genes else 0 for g in target_genes_upper]
        matrix_data.append(row)
        pathway_names.append(name)

    # 先构建初步的 DataFrame
    adj_matrix = pd.DataFrame(
        matrix_data, 
        index=pathway_names, 
        columns=target_genes,
        dtype=np.int8
    )

    # 3. 核心逻辑：增加一个维度标识“无通路基因”
    # 检查哪些列（基因）的和等于 0
    # sum(axis=0) 对列求和，得到每个基因参与的通路总数
    gene_sums = adj_matrix.sum(axis=0)
    
    # 定义新的一行：如果没有被任何通路覆盖（sum == 0），则置为 1，否则为 0
    orphan_row = (gene_sums == 0).astype(np.int8)
    
    # 将这一行命名为 'NO_ANNOTATED_GO_BP' 并添加到矩阵底部
    adj_matrix.loc['NO_ANNOTATED_GO'] = orphan_row

    return adj_matrix

# --- 执行 ---
gmt_file = "/home/featurize/work/cs-main/DeepTGI/dataset/scRNA-Seq/BP/c5.go.mf.v2026.1.Hs.symbols.gmt"
my_genes = column_names

adj_df = gmt_to_binary_matrix_with_orphan_flag(gmt_file, my_genes)

# 验证
orphan_count = adj_df.loc['NO_ANNOTATED_GO'].sum()
print(f"处理完成！共有 {orphan_count} 个基因没有任何 GO 注释，已标记在 'NO_ANNOTATED_GO' 维度。")

# 保存结果
# 建议保留 index，因为最后一行 'NO_ANNOTATED_GO_BP' 是一个重要的特征维度
print(adj_df.shape)
adj_df.to_csv("hESC_MF.csv",index=False)

910
解析完成，共包含 1872 条通路。
处理完成！共有 140 个基因没有任何 GO 注释，已标记在 'NO_ANNOTATED_GO' 维度。
(1873, 910)
